# Bias Detection & Fairness Analysis

This notebook implements bias detection and fairness analysis for the NovaCred credit application dataset.
As part of the Data Governance Task Force, the goal is to identify potential discrimination in historical
lending decisions and quantify bias using established fairness metrics.

**Analyses performed:**
1. Gender Bias — Disparate Impact Ratio (four-fifths rule)
2. Statistical Significance Testing — Chi-squared test
3. Fairlearn Fairness Metrics — Demographic parity difference
4. Rejection Reason Analysis — Gender-based rejection patterns
5. Proxy Discrimination — Spending categories and income as gender proxies
6. Intersectional Analysis — Approval patterns across income brackets and gender

## 1. Setup & Data Loading

We load the consolidated dataset produced by the Data Engineering pipeline. This dataset
merges the original application data with pivoted spending behavior into a single file.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from fairlearn.metrics import (
    demographic_parity_difference,
    demographic_parity_ratio,
    MetricFrame,
    selection_rate
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully.')

In [ ]:
# Load consolidated dataset (spending behavior merged into main applications)
df = pd.read_csv('../data/processed/cleaned_credit_applications.csv')

print(f'Dataset shape: {df.shape[0]} records x {df.shape[1]} columns')
print(f'\nColumns:')
for i, col in enumerate(df.columns):
    print(f'  {i+1:2d}. {col}')
print(f'\nGender distribution:')
print(df['applicant_info.gender'].value_counts(dropna=False))
print(f'\nOverall approval rate: {df["decision.loan_approved"].mean():.1%}')

In [ ]:
# Filter to Male/Female for binary fairness analysis
# 2 records with 'Unknown' gender are excluded
df_bias = df[df['applicant_info.gender'].isin(['Male', 'Female'])].copy()
df_bias['gender_binary'] = (df_bias['applicant_info.gender'] == 'Male').astype(int)
df_bias['approved_binary'] = df_bias['decision.loan_approved'].astype(int)

print(f'Records for bias analysis: {len(df_bias)} (excluded {len(df) - len(df_bias)} with Unknown gender)')
print(f'Female: {len(df_bias[df_bias["applicant_info.gender"]=="Female"])}')
print(f'Male:   {len(df_bias[df_bias["applicant_info.gender"]=="Male"])}')

---
## 2. Disparate Impact Ratio (DIR) — Four-Fifths Rule

The **Disparate Impact Ratio** measures whether a selection process disproportionately affects a protected group.
Under the EEOC's **four-fifths (80%) rule**, a selection rate for any group less than 80% of the highest-performing
group constitutes evidence of adverse impact.

$$DIR = \frac{\text{Approval Rate}_{\text{unprivileged}}}{\text{Approval Rate}_{\text{privileged}}}$$

- **DIR < 0.8** → Potential disparate impact (violates four-fifths rule)
- **DIR = 1.0** → Perfect parity

In [ ]:
# Calculate approval rates by gender
approval_rates = df_bias.groupby('applicant_info.gender')['decision.loan_approved'].agg(['mean', 'sum', 'count'])
approval_rates.columns = ['Approval Rate', 'Approved Count', 'Total Count']
approval_rates['Rejected Count'] = approval_rates['Total Count'] - approval_rates['Approved Count']

print('=== Approval Statistics by Gender ===')
print(approval_rates.to_string())

# Compute Disparate Impact Ratio
rate_female = approval_rates.loc['Female', 'Approval Rate']
rate_male = approval_rates.loc['Male', 'Approval Rate']
DIR = rate_female / rate_male

print(f'\n{"="*50}')
print(f'Female Approval Rate: {rate_female:.1%} ({int(approval_rates.loc["Female","Approved Count"])}/{int(approval_rates.loc["Female","Total Count"])})')
print(f'Male Approval Rate:   {rate_male:.1%} ({int(approval_rates.loc["Male","Approved Count"])}/{int(approval_rates.loc["Male","Total Count"])})')
print(f'\nDisparate Impact Ratio (DIR): {DIR:.4f}')
print(f'Four-Fifths Threshold:         0.8000')
print(f'\nVERDICT: {"DISPARATE IMPACT DETECTED" if DIR < 0.8 else "No disparate impact"}')
print(f'The female approval rate is {DIR:.1%} of the male rate, which is')
print(f'{"BELOW" if DIR < 0.8 else "above"} the 80% threshold.')

In [ ]:
# Visualize approval rates with DIR threshold
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of approval rates
colors = ['#e74c3c', '#3498db']
bars = axes[0].bar(['Female', 'Male'], [rate_female, rate_male], color=colors, edgecolor='black', linewidth=0.8)
axes[0].axhline(y=rate_male * 0.8, color='red', linestyle='--', linewidth=2, label=f'80% of Male rate ({rate_male*0.8:.1%})')
axes[0].set_ylabel('Approval Rate', fontsize=12)
axes[0].set_title('Loan Approval Rate by Gender', fontsize=14, fontweight='bold')
axes[0].set_ylim(0, 1)
axes[0].legend(fontsize=10)
for bar, rate in zip(bars, [rate_female, rate_male]):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                f'{rate:.1%}', ha='center', va='bottom', fontsize=14, fontweight='bold')

# Stacked bar chart: approved vs denied
approval_data = approval_rates[['Approved Count', 'Rejected Count']]
approval_data.plot(kind='bar', stacked=True, ax=axes[1], color=['#2ecc71', '#e74c3c'], edgecolor='black')
axes[1].set_title('Approved vs Denied by Gender', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Number of Applications')
axes[1].set_xlabel('Gender')
axes[1].legend(['Approved', 'Denied'])
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('../reports/gender_approval_rates.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\nDisparate Impact Ratio = {DIR:.4f} (threshold: 0.8)')

---
## 3. Statistical Significance Testing

We perform a **Chi-squared test of independence** to determine whether the relationship between
gender and loan approval is statistically significant (not due to chance).

- **H₀ (Null):** Gender and loan approval are independent
- **H₁ (Alternative):** Gender and loan approval are associated
- **Significance level:** α = 0.05

In [ ]:
# Contingency table
contingency = pd.crosstab(
    df_bias['applicant_info.gender'],
    df_bias['decision.loan_approved'],
    margins=True
)
contingency.columns = ['Rejected', 'Approved', 'Total']
contingency.index = ['Female', 'Male', 'Total']
print('=== Contingency Table ===')
print(contingency)

# Chi-squared test
ct = pd.crosstab(df_bias['applicant_info.gender'], df_bias['decision.loan_approved'])
chi2, p_value, dof, expected = stats.chi2_contingency(ct)

print(f'\n=== Chi-Squared Test of Independence ===')
print(f'Chi² statistic:     {chi2:.4f}')
print(f'p-value:            {p_value:.6f}')
print(f'Degrees of freedom: {dof}')

expected_df = pd.DataFrame(expected, index=['Female', 'Male'], columns=['Rejected', 'Approved'])
print(f'\nExpected frequencies (if independent):')
print(expected_df.round(1))

print(f'\n{"="*50}')
print(f'RESULT: The association is {"STATISTICALLY SIGNIFICANT" if p_value < 0.05 else "not significant"} (p = {p_value:.6f})')
if p_value < 0.05:
    print(f'We reject H₀: gender and loan approval are NOT independent.')

# Effect size: Cramér's V
n = len(df_bias)
cramers_v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
effect = 'negligible' if cramers_v < 0.1 else 'small-to-moderate' if cramers_v < 0.3 else 'moderate' if cramers_v < 0.5 else 'large'
print(f'\nEffect Size (Cramér\'s V): {cramers_v:.4f} ({effect})')

In [ ]:
# Visualize observed vs expected
fig, ax = plt.subplots(figsize=(8, 5))

x = np.arange(2)
width = 0.35

observed_female = [ct.loc['Female', False], ct.loc['Female', True]]
observed_male = [ct.loc['Male', False], ct.loc['Male', True]]

bars1 = ax.bar(x - width/2, observed_female, width, label='Female (Observed)', color='#e74c3c', alpha=0.8)
bars2 = ax.bar(x + width/2, observed_male, width, label='Male (Observed)', color='#3498db', alpha=0.8)

# Expected values as dashed outlines
ax.bar(x - width/2, [expected_df.loc['Female', 'Rejected'], expected_df.loc['Female', 'Approved']],
       width, label='Female (Expected)', fill=False, edgecolor='#e74c3c', linewidth=2, linestyle='--')
ax.bar(x + width/2, [expected_df.loc['Male', 'Rejected'], expected_df.loc['Male', 'Approved']],
       width, label='Male (Expected)', fill=False, edgecolor='#3498db', linewidth=2, linestyle='--')

ax.set_xticks(x)
ax.set_xticklabels(['Rejected', 'Approved'], fontsize=12)
ax.set_ylabel('Number of Applicants', fontsize=12)
ax.set_title(f'Observed vs. Expected Counts by Gender\n(Chi² = {chi2:.2f}, p = {p_value:.4f})', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('../reports/chi_squared_test.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Fairlearn Fairness Metrics

Using Microsoft's [Fairlearn](https://fairlearn.org/) library, we compute standardized fairness metrics:

- **Demographic Parity Difference (DPD):** Difference in selection rates between groups. Ideal = 0.
- **Demographic Parity Ratio (DPR):** Ratio of selection rates. Ideal = 1. (Equivalent to DIR.)

In [ ]:
y_true = df_bias['approved_binary'].values
sensitive = df_bias['applicant_info.gender'].values

# The loan decision IS the model output — we treat it as both y_true and y_pred
y_pred = y_true

# Demographic Parity Difference
dpd = demographic_parity_difference(y_true=y_true, y_pred=y_pred, sensitive_features=sensitive)

# Demographic Parity Ratio (equivalent to DIR)
dpr = demographic_parity_ratio(y_true=y_true, y_pred=y_pred, sensitive_features=sensitive)

print('=== Fairlearn Fairness Metrics ===')
print(f'Demographic Parity Difference (DPD): {dpd:.4f}')
print(f'  Ideal = 0 | Observed gap = {abs(dpd):.1%} difference in approval rates')
print(f'  {"UNFAIR" if abs(dpd) > 0.1 else "Within acceptable range"} (threshold: |DPD| > 0.10)\n')

print(f'Demographic Parity Ratio (DPR):      {dpr:.4f}')
print(f'  Ideal = 1.0 | Matches DIR calculation: {DIR:.4f}')
print(f'  {"UNFAIR" if dpr < 0.8 else "Within acceptable range"} (four-fifths rule: DPR < 0.80)')

# Per-group breakdown using MetricFrame
mf = MetricFrame(
    metrics={'selection_rate': selection_rate, 'count': lambda y_true, y_pred: len(y_true)},
    y_true=y_true, y_pred=y_true, sensitive_features=sensitive
)
print(f'\nPer-group metrics:')
print(mf.by_group)
print(f'\nOverall selection rate: {mf.overall["selection_rate"]:.4f}')

---
## 5. Rejection Reason Analysis

Examining the distribution of rejection reasons by gender to understand where the algorithm
disproportionately rejects female applicants.

In [ ]:
# Rejection reason breakdown by gender
rejected = df_bias[df_bias['decision.loan_approved'] == False].copy()

rejection_ct = pd.crosstab(rejected['applicant_info.gender'], rejected['decision.rejection_reason'], margins=True)
print('=== Rejection Reasons by Gender (Counts) ===')
print(rejection_ct)

# Proportions within each gender
print('\n=== Rejection Reason Proportions (within each gender) ===')
rejection_props = pd.crosstab(rejected['applicant_info.gender'], rejected['decision.rejection_reason'], normalize='index')
print(rejection_props.round(3))

In [ ]:
# Visualize rejection reasons
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Absolute counts
rejection_counts = pd.crosstab(rejected['applicant_info.gender'], rejected['decision.rejection_reason'])
rejection_counts.plot(kind='bar', ax=axes[0], color=['#e74c3c', '#f39c12', '#3498db', '#2ecc71'], edgecolor='black')
axes[0].set_title('Rejection Reasons by Gender (Counts)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Gender')
axes[0].set_ylabel('Number of Rejections')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(title='Reason', fontsize=9)

# As rate of total applicants per gender
reasons = rejected.groupby(['applicant_info.gender', 'decision.rejection_reason']).size().unstack(fill_value=0)
totals = df_bias.groupby('applicant_info.gender').size()
rejection_rate_by_reason = reasons.div(totals, axis=0)
rejection_rate_by_reason.plot(kind='bar', ax=axes[1], color=['#e74c3c', '#f39c12', '#3498db', '#2ecc71'], edgecolor='black')
axes[1].set_title('Rejection Rate by Reason (% of Total Applicants)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Gender')
axes[1].set_ylabel('Rejection Rate')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Reason', fontsize=9)

plt.tight_layout()
plt.savefig('../reports/rejection_reasons.png', dpi=150, bbox_inches='tight')
plt.show()

# Key finding
algo_f = rejection_counts.loc['Female', 'algorithm_risk_score'] if 'algorithm_risk_score' in rejection_counts.columns else 0
algo_m = rejection_counts.loc['Male', 'algorithm_risk_score'] if 'algorithm_risk_score' in rejection_counts.columns else 0
total_f = len(df_bias[df_bias['applicant_info.gender']=='Female'])
total_m = len(df_bias[df_bias['applicant_info.gender']=='Male'])
print(f'Key Finding: \"algorithm_risk_score\" rejected {algo_f} females ({algo_f/total_f:.1%}) vs {algo_m} males ({algo_m/total_m:.1%})')

---
## 6. Proxy Discrimination Analysis

**Proxy discrimination** occurs when non-protected attributes are correlated with a protected
characteristic (gender) and are used in decision-making, effectively encoding the protected
attribute indirectly.

We investigate:
1. Which features correlate with **both** gender and approval
2. Whether spending patterns differ systematically by gender
3. Whether approval rates differ within comparable financial profiles

In [ ]:
# 6.1 Correlation analysis: features vs gender and vs approval
spending_cols = [c for c in df_bias.columns if c.startswith('spending_')]
financial_cols = ['financials.annual_income', 'financials.credit_history_months',
                  'financials.debt_to_income', 'financials.savings_balance']
analysis_cols = financial_cols + spending_cols

correlations = []
for col in analysis_cols:
    corr_gender = df_bias[['gender_binary', col]].corr().iloc[0, 1]
    corr_approval = df_bias[['approved_binary', col]].corr().iloc[0, 1]
    correlations.append({
        'Feature': col.replace('spending_', 'Spending: ').replace('financials.', ''),
        'Corr with Gender (Male=1)': round(corr_gender, 4),
        'Corr with Approval': round(corr_approval, 4),
        'Proxy Risk': 'HIGH' if abs(corr_gender) > 0.05 and abs(corr_approval) > 0.05 else
                      'MODERATE' if abs(corr_gender) > 0.05 or abs(corr_approval) > 0.05 else 'LOW'
    })

corr_df = pd.DataFrame(correlations).sort_values('Proxy Risk', ascending=True)
print('=== Feature Correlation Analysis ===')
print(corr_df.to_string(index=False))

In [ ]:
# 6.2 Proxy scatter plot: correlation with gender vs correlation with approval
fig, ax = plt.subplots(figsize=(12, 8))

from matplotlib.patches import Patch
from matplotlib.lines import Line2D

for _, row in corr_df.iterrows():
    color = {'HIGH': '#e74c3c', 'MODERATE': '#f39c12', 'LOW': '#2ecc71'}[row['Proxy Risk']]
    marker = 's' if not row['Feature'].startswith('Spending') else 'o'
    ax.scatter(row['Corr with Gender (Male=1)'], row['Corr with Approval'],
              c=color, s=120, marker=marker, edgecolors='black', linewidth=0.5, zorder=5)
    ax.annotate(row['Feature'].replace('Spending: ', ''), 
               (row['Corr with Gender (Male=1)'], row['Corr with Approval']),
               textcoords='offset points', xytext=(5, 5), fontsize=8)

ax.axhspan(-0.05, 0.05, color='gray', alpha=0.1)
ax.axvspan(-0.05, 0.05, color='gray', alpha=0.1)
ax.axhline(y=0, color='black', linewidth=0.5)
ax.axvline(x=0, color='black', linewidth=0.5)

legend_elements = [
    Patch(facecolor='#e74c3c', label='HIGH proxy risk'),
    Patch(facecolor='#f39c12', label='MODERATE proxy risk'),
    Patch(facecolor='#2ecc71', label='LOW proxy risk'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='gray', markersize=10, label='Financial feature'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='gray', markersize=10, label='Spending feature'),
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=10)
ax.set_xlabel('Correlation with Gender (Male=1)', fontsize=12)
ax.set_ylabel('Correlation with Approval', fontsize=12)
ax.set_title('Proxy Discrimination Map\nFeatures in corners (correlated with BOTH gender and approval) are potential proxies',
            fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/proxy_discrimination_map.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 6.3 Spending patterns by gender — heatmap
spending_by_gender = df_bias.groupby('applicant_info.gender')[spending_cols].mean()
spending_by_gender.columns = [c.replace('spending_', '') for c in spending_cols]

fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(spending_by_gender, annot=True, fmt='.0f', cmap='RdYlBu_r', ax=ax,
           linewidths=0.5, linecolor='white')
ax.set_title('Average Spending by Category and Gender', fontsize=14, fontweight='bold')
ax.set_ylabel('')

plt.tight_layout()
plt.savefig('../reports/spending_by_gender.png', dpi=150, bbox_inches='tight')
plt.show()

# Highlight biggest differences
print('=== Largest Spending Differences by Gender ===')
diff = spending_by_gender.loc['Female'] - spending_by_gender.loc['Male']
diff_sorted = diff.abs().sort_values(ascending=False)
for cat in diff_sorted.head(5).index:
    f_val = spending_by_gender.loc['Female', cat]
    m_val = spending_by_gender.loc['Male', cat]
    print(f'  {cat}: Female={f_val:.0f}, Male={m_val:.0f} (diff={f_val-m_val:+.0f})')

In [ ]:
# 6.4 Approval rates within comparable income brackets (controlling for income)
df_bias['income_bracket'] = pd.cut(
    df_bias['financials.annual_income'],
    bins=[0, 50000, 75000, 100000, 200000],
    labels=['<50k', '50-75k', '75-100k', '100k+']
)

bracket_rates = df_bias.groupby(['income_bracket', 'applicant_info.gender'])['approved_binary'].agg(['mean', 'count']).unstack()
print('=== Approval Rates by Income Bracket and Gender ===')
print(bracket_rates.round(3))

# Visualize
fig, ax = plt.subplots(figsize=(10, 6))
brackets = ['<50k', '50-75k', '75-100k', '100k+']
female_rates = [bracket_rates.loc[b, ('mean', 'Female')] for b in brackets]
male_rates = [bracket_rates.loc[b, ('mean', 'Male')] for b in brackets]
x = np.arange(len(brackets))
width = 0.35

ax.bar(x - width/2, female_rates, width, label='Female', color='#e74c3c', edgecolor='black', linewidth=0.8)
ax.bar(x + width/2, male_rates, width, label='Male', color='#3498db', edgecolor='black', linewidth=0.8)

# DIR annotation for each bracket
for i, (fr, mr) in enumerate(zip(female_rates, male_rates)):
    di = fr / mr if mr > 0 else 0
    ax.annotate(f'DIR={di:.2f}', (i, max(fr, mr) + 0.03), ha='center', fontsize=10,
               color='red' if di < 0.8 else 'green', fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(brackets, fontsize=12)
ax.set_ylabel('Approval Rate', fontsize=12)
ax.set_xlabel('Annual Income Bracket', fontsize=12)
ax.set_title('Approval Rate by Income Bracket and Gender\n(DIR annotated per bracket)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('../reports/income_bracket_bias.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nFemale applicants have lower approval rates across ALL income brackets.')
print('The disparity is NOT explained by income differences alone.')

In [ ]:
# 6.5 Financial profile comparison — are female applicants objectively riskier?
print('=== Financial Profile Comparison by Gender ===')
print('If the approval disparity is fair, female applicants should have worse financial profiles.\n')

for col in financial_cols:
    col_short = col.replace('financials.', '')
    f_vals = df_bias[df_bias['applicant_info.gender']=='Female'][col].dropna()
    m_vals = df_bias[df_bias['applicant_info.gender']=='Male'][col].dropna()
    f_mean, m_mean = f_vals.mean(), m_vals.mean()
    diff_pct = ((f_mean - m_mean) / m_mean) * 100 if m_mean != 0 else 0
    t_stat, t_pval = stats.ttest_ind(f_vals, m_vals)
    sig = '*' if t_pval < 0.05 else ''
    
    print(f'{col_short}:')
    print(f'  Female mean: {f_mean:.2f} | Male mean: {m_mean:.2f} | Diff: {diff_pct:+.1f}% | p={t_pval:.4f} {sig}')

print('\n* = statistically significant difference at p < 0.05')
print('\nConclusion: Financial profiles are largely comparable between genders,')
print('yet approval rates differ significantly — supporting a finding of algorithmic bias.')